# 5.Module recommender: select meals based on KBZHU

## 1.Importing Libraries

In [33]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator

all_recipes = pd.read_csv('../data/clean/recipes_clean.csv')

kazakh_recipes = all_recipes[all_recipes['source'] == 'kazakh'].copy().reset_index(drop = True)
foodcom_recipes = all_recipes[all_recipes['source'] == 'foodcom'].copy().reset_index(drop = True)

print('kazakh meals(russian): ', len(kazakh_recipes))
print('foodcom meals(english): ', len(foodcom_recipes))
print()
print('Example of kz meals:')
print(kazakh_recipes[['name','calories','protein','fat','carbs','minutes']].head(5))

kazakh meals(russian):  142
foodcom meals(english):  217511

Example of kz meals:
             name  calories  protein   fat  carbs  minutes
0       Бешбармак     950.0     62.5  37.5   87.5    210.0
1           Манты     462.0     25.0  17.5   50.0    100.0
2          Лагман     275.0     17.5  15.0   37.5     80.0
3         Куырдак     475.0     35.0  27.5   27.5    180.0
4  Плов с курицей     575.0     27.5  22.5   60.0     60.0


## 2.Translator function

In [34]:
def translate_to_russian(text):
    try:
        translator = GoogleTranslator(source='en', target='ru')
        return translator.translate(str(text))
    except:
        return text


def translate_ingredients(ingredients_str):
    try:
        if isinstance(ingredients_str, str) and ingredients_str.startswith('['):
            items = ast.literal_eval(ingredients_str)
        else:
            items = [i.strip() for i in str(ingredients_str).split(',')]
        translated = [translate_to_russian(item) for item in items[:8]]  # первые 8
        return ', '.join(translated)
    except:
        return ingredients_str

print("Translation:")
print('chicken breast with rice ->' , translate_to_russian('chicken breast with rice'))
print('scrumbled eggs ->', translate_to_russian('scrumbled eggs'))

Translation:
chicken breast with rice -> куриная грудка с рисом
scrumbled eggs -> яичница


## 3.Filtering by fridge

In [35]:
def filter_by_fridge(recipes_df, fridge_items, min_match=0.25):
    if not fridge_items:
        return recipes_df

    fridge_lower = [item.lower().strip() for item in fridge_items]

    def match_score(ingredients_str):
        try:
            if isinstance(ingredients_str, str) and ingredients_str.startswith('['):
                ingredients = ast.literal_eval(ingredients_str)
            else:
                ingredients = [i.strip() for i in str(ingredients_str).split(',')]

            ingredients_lower = [i.lower().strip() for i in ingredients]
            matches = sum(
                any(fridge_item in ing or ing in fridge_item
                    for fridge_item in fridge_lower)
                for ing in ingredients_lower
            )
            return matches / max(len(ingredients_lower), 1)
        except:
            return 0

    df = recipes_df.copy()
    df['match_score'] = df['ingredients'].apply(match_score)
    filtered = df[df['match_score'] >= min_match]

    if len(filtered) < 5:
        filtered = df.nlargest(30, 'match_score')

    return filtered.reset_index(drop=True)


print('Filtering function is created')

Filtering function is created


## 4.Function recommendation by cosine similarity

In [37]:
def recommend_from_dataset(target_cal, target_prot, target_fat,
                           target_carbs, max_minutes, recipes_df, top_n=3,
                           exclude_names=None):

    df = recipes_df.copy()
    df = df[df['minutes'] <= max_minutes]
    df = df.dropna(subset=['calories','protein','fat','carbs'])

    # Исключаем уже показанные блюда
    if exclude_names:
        df = df[~df['name'].isin(exclude_names)]

    if len(df) == 0:
        return pd.DataFrame()

    features = ['calories','protein','fat','carbs','minutes']
    sc = MinMaxScaler()
    recipe_vecs = sc.fit_transform(df[features])

    target = np.array([[target_cal, target_prot, target_fat, target_carbs, max_minutes]])
    target_vec = sc.transform(target)

    sims = cosine_similarity(target_vec, recipe_vecs)[0]
    top_idx = sims.argsort()[-top_n:][::-1]

    result = df.iloc[top_idx].copy()
    result['similarity'] = sims[top_idx].round(4)
    return result

print('Функция recommend_from_dataset создана')


Функция recommend_from_dataset создана


## 5.Main function(priority to kazakh dataset)

In [38]:
def recommend_meal(target_cal, target_prot, target_fat, target_carbs,
                   max_minutes, fridge_items=None, top_n=3, exclude_names=None):

    results = []

    kazakh_filtered = filter_by_fridge(kazakh_recipes, fridge_items) \
                      if fridge_items else kazakh_recipes.copy()

    kazakh_time = kazakh_filtered[kazakh_filtered['minutes'] <= max_minutes]

    kazakh_recs = recommend_from_dataset(
        target_cal, target_prot, target_fat, target_carbs,
        max_minutes, kazakh_time, top_n=top_n,
         exclude_names=exclude_names
    ) if len(kazakh_time) > 0 else pd.DataFrame()

    for _, row in kazakh_recs.iterrows():
        results.append({
            'name':        row['name'],
            'ingredients': row['ingredients'],
            'calories':    round(row['calories']),
            'protein':     round(row['protein']),
            'fat':         round(row['fat']),
            'carbs':       round(row['carbs']),
            'minutes':     int(row['minutes']),
            'similarity':  row['similarity'],
            'language':    'ru'
        })

    need_more = top_n - len(results)
    if need_more > 0:
        foodcom_filtered = filter_by_fridge(foodcom_recipes, fridge_items) \
                           if fridge_items else foodcom_recipes.copy()

        foodcom_recs = recommend_from_dataset(
            target_cal, target_prot, target_fat, target_carbs,
            max_minutes, foodcom_filtered, top_n=need_more,
            exclude_names=exclude_names
        )

        if len(foodcom_recs) > 0:
            print(f'  Переводим {len(foodcom_recs)} блюд из Food.com...')
            for _, row in foodcom_recs.iterrows():
                results.append({
                    'name':        translate_to_russian(row['name']),
                    'ingredients': translate_ingredients(row['ingredients']),
                    'calories':    round(row['calories']),
                    'protein':     round(row['protein']),
                    'fat':         round(row['fat']),
                    'carbs':       round(row['carbs']),
                    'minutes':     int(row['minutes']),
                    'similarity':  row['similarity'],
                    'language':    'ru (переведено)'
                })

    return results

## 6.Function of breaking down the KBZhU by meals

In [39]:
def split_daily_kbzhu(calories, protein, fat, carbs):
    rations =  {
        'Breakfast': 0.25 ,
        'Lunch': 0.35 ,
        'Dinner': 0.30 ,
        'Snack': 0.10 ,
    }
    max_times = {
        'Breakfast': 20 ,
        'Lunch': 60 ,
        'Dinner': 45 ,
        'Snack': 10 ,
    }
    result = { }
    for meal, ratio in rations.items():
        result[meal] = {
            'calories': round(calories * ratio),
            'protein': round(protein * ratio),
            'fat': round(fat * ratio),
            'carbs': round(carbs * ratio),
            'max_minutes': max_times[meal],
        }
    return result

print("Split daily KBZHU is created")

Split daily KBZHU is created


## 7.Full test of the recommendator

In [40]:
daily_kbzhu = {
    'calories': 2200,
    'protein':  110,
    'fat':       70,
    'carbs':    270
}

fridge_ru = ['яйцо', 'молоко', 'курица', 'рис', 'лук',
             'масло', 'помидор', 'картофель', 'морковь']

meal_split = split_daily_kbzhu(
    daily_kbzhu['calories'], daily_kbzhu['protein'],
    daily_kbzhu['fat'], daily_kbzhu['carbs']
)

print('РЕКОМЕНДАЦИИ НА ДЕНЬ')
print(f"Суточная норма: {daily_kbzhu['calories']} ккал | "
      f"Б:{daily_kbzhu['protein']}г | "
      f"Ж:{daily_kbzhu['fat']}г | "
      f"У:{daily_kbzhu['carbs']}г")

shown_today = []

for meal_name, kbzhu in meal_split.items():
    print(f'\n{meal_name} (цель: {kbzhu["calories"]} ккал | '
          f'макс {kbzhu["max_minutes"]} мин):')

    recs = recommend_meal(
        target_cal   = kbzhu['calories'],
        target_prot  = kbzhu['protein'],
        target_fat   = kbzhu['fat'],
        target_carbs = kbzhu['carbs'],
        max_minutes  = kbzhu['max_minutes'],
        fridge_items = fridge_ru,
        top_n        = 3,
        exclude_names = shown_today
    )
    for i, rec in enumerate(recs, 1):
        target_cal_meal = kbzhu['calories']
        rec_cal = rec['calories']

        if rec_cal < target_cal_meal * 0.7:
            portions = round(target_cal_meal / rec_cal, 1)
            portion_note = f' → рекомендуем {portions} порции = {round(rec_cal * portions)} ккал'
        else:
            portion_note = ''

        print(f'  {i}. {rec["name"]}')
        print(f'     {rec["calories"]} ккал | Б:{rec["protein"]}г | '
              f'Ж:{rec["fat"]}г | У:{rec["carbs"]}г | '
              f'{rec["minutes"]} мин')
        if portion_note:
            print(f'  ️ Калорий мало{portion_note}')
        shown_today.append(rec['name'])

РЕКОМЕНДАЦИИ НА ДЕНЬ
Суточная норма: 2200 ккал | Б:110г | Ж:70г | У:270г

Breakfast (цель: 550 ккал | макс 20 мин):
  1. Бутерброд с яйцом
     260 ккал | Б:12г | Ж:12г | У:26г | 8 мин
  ️ Калорий мало → рекомендуем 2.1 порции = 546 ккал
  2. Пельмени со сметаной
     480 ккал | Б:20г | Ж:22г | У:48г | 20 мин
  3. Сырники со сметаной
     420 ккал | Б:22г | Ж:18г | У:38г | 20 мин

Lunch (цель: 770 ккал | макс 60 мин):
  1. Макароны по-флотски
     480 ккал | Б:24г | Ж:18г | У:54г | 30 мин
  ️ Калорий мало → рекомендуем 1.6 порции = 768 ккал
  2. Рис с курицей
     450 ккал | Б:28г | Ж:12г | У:52г | 40 мин
  ️ Калорий мало → рекомендуем 1.7 порции = 765 ккал
  3. Жареные котлеты с рисом
     580 ккал | Б:28г | Ж:24г | У:56г | 40 мин

Dinner (цель: 660 ккал | макс 45 мин):
  1. Крабовый салат
     260 ккал | Б:12г | Ж:12г | У:26г | 15 мин
  ️ Калорий мало → рекомендуем 2.5 порции = 650 ккал
  2. Макароны с сыром
     460 ккал | Б:16г | Ж:18г | У:56г | 20 мин
  ️ Калорий мало → рекомендуе

In [41]:
import pickle

recommender_data = {
    'kazakh_recipes':  kazakh_recipes,
    'foodcom_recipes': foodcom_recipes,
    'features':        ['calories','protein','fat','carbs','minutes']
}

with open('../models/recommender_vectors.pkl', 'wb') as f:
    pickle.dump(recommender_data, f)

print('Рекомендатор сохранён: models/recommender_vectors.pkl')
print('Казахских блюд:', len(kazakh_recipes))
print('Food.com блюд:', len(foodcom_recipes))

Рекомендатор сохранён: models/recommender_vectors.pkl
Казахских блюд: 142
Food.com блюд: 217511


In [43]:
import pickle
import pandas as pd

# Загружаем НОВЫЙ казахский датасет с category
kazakh_recipes = pd.read_csv('../data/raw/kazakh_recipes.csv')
kazakh_recipes['source'] = 'kazakh'

# Перезагружаем foodcom
foodcom_recipes = pd.read_csv('../data/clean/recipes_clean.csv')
foodcom_recipes = foodcom_recipes[foodcom_recipes['source'] == 'foodcom'].copy()

recommender_data = {
    'kazakh_recipes':  kazakh_recipes,
    'foodcom_recipes': foodcom_recipes,
    'features':        ['calories','protein','fat','carbs','minutes']
}

with open('../models/recommender_vectors.pkl', 'wb') as f:
    pickle.dump(recommender_data, f)

print('Пересохранено!')
print('Казахских блюд:', len(kazakh_recipes))
print('Колонки:', kazakh_recipes.columns.tolist())
print(kazakh_recipes['category'].value_counts())

Пересохранено!
Казахских блюд: 142
Колонки: ['name', 'ingredients', 'minutes', 'calories', 'protein', 'fat', 'carbs', 'category', 'source']
category
main         53
snack        33
breakfast    23
soup         18
salad        15
Name: count, dtype: int64
